# Alignment: DPO and ORPO Training

Hands-on examples for Module 07, covering:

1. Preference dataset creation
2. DPO training with TRL
3. ORPO training (no reference model)
4. Comparing SFT vs. DPO outputs
5. Preference evaluation

## Setup

In [ ]:
# Install required packages
# !pip install transformers trl datasets accelerate torch

import torch
import matplotlib.pyplot as plt
from datasets import Dataset

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Creating Preference Datasets

In [ ]:
# Sample preference data (in practice, load from file or API)
preference_data = [
    {
        "prompt": "How do I stay motivated?",
        "chosen": "Here are some evidence-based strategies: 1) Set clear, achievable goals. 2) Break tasks into smaller steps. 3) Track your progress. 4) Celebrate small wins. 5) Find an accountability partner.",
        "rejected": "Just push through it."
    },
    {
        "prompt": "Explain quantum entanglement",
        "chosen": "Quantum entanglement is when two particles become linked so that measuring one instantly affects the other, regardless of distance. Einstein called it 'spooky action at a distance.' It's been experimentally verified and is key to quantum computing.",
        "rejected": "Particles that are connected somehow."
    },
    {
        "prompt": "What's the best programming language?",
        "chosen": "There's no single 'best' language—it depends on your goals. Python is great for beginners and data science. JavaScript is essential for web development. Rust offers memory safety for systems programming. What kind of projects interest you?",
        "rejected": "Python is the best, everyone should use it."
    },
    {
        "prompt": "I'm feeling sad, what should I do?",
        "chosen": "I'm sorry you're going through this. It might help to: talk to someone you trust, get some fresh air, practice self-compassion. If these feelings persist, consider speaking with a mental health professional. You're not alone.",
        "rejected": "Cheer up! It could be worse."
    },
    {
        "prompt": "Write a haiku about coding",
        "chosen": "Silent keys clicking\nLogic flows through fingertips\nBugs hide, then emerge",
        "rejected": "Code code code all day\nComputer says things sometimes\nProgramming is code"
    }
]

# Convert to Hugging Face Dataset
pref_dataset = Dataset.from_list(preference_data)
print(f"Created preference dataset with {len(pref_dataset)} samples")
print(f"\nSample entry:")
print(pref_dataset[0])

In [ ]:
# Analyze preference data quality
def analyze_preferences(dataset):
    """Analyze preference dataset statistics."""
    chosen_lengths = [len(d["chosen"]) for d in dataset]
    rejected_lengths = [len(d["rejected"]) for d in dataset]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Length comparison
    axes[0].hist(chosen_lengths, alpha=0.7, label="Chosen", bins=10)
    axes[0].hist(rejected_lengths, alpha=0.7, label="Rejected", bins=10)
    axes[0].set_xlabel("Response Length (chars)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Response Length Distribution")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Length ratio
    ratios = [c/r if r > 0 else 0 for c, r in zip(chosen_lengths, rejected_lengths)]
    axes[1].bar(range(len(ratios)), ratios, color='#238636')
    axes[1].axhline(y=1, color='red', linestyle='--', label='Equal length')
    axes[1].set_xlabel("Sample")
    axes[1].set_ylabel("Chosen/Rejected Length Ratio")
    axes[1].set_title("Length Ratio per Sample")
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Average chosen length: {sum(chosen_lengths)/len(chosen_lengths):.0f} chars")
    print(f"Average rejected length: {sum(rejected_lengths)/len(rejected_lengths):.0f} chars")
    print(f"Average ratio: {sum(ratios)/len(ratios):.2f}x")

analyze_preferences(pref_dataset)

## 2. DPO Training with TRL

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import DPOTrainer, DPOConfig

# Use a small model for demonstration
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

ref_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Tokenize preference dataset
def tokenize_pair(example):
    """Tokenize prompt, chosen, and rejected."""
    tokenized_chosen = tokenizer(
        example["chosen"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    tokenized_rejected = tokenizer(
        example["rejected"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )
    
    # Add prompt for context
    tokenized_prompt = tokenizer(
        example["prompt"],
        truncation=True,
        max_length=128
    )
    
    return {
        "input_ids_chosen": tokenized_chosen["input_ids"],
        "attention_mask_chosen": tokenized_chosen["attention_mask"],
        "input_ids_rejected": tokenized_rejected["input_ids"],
        "attention_mask_rejected": tokenized_rejected["attention_mask"],
    }

tokenized_pref = pref_dataset.map(tokenize_pair, batched=False)
tokenized_pref = tokenized_pref.remove_columns(["prompt", "chosen", "rejected"])
tokenized_pref.set_format(type="torch")

print(f"Tokenized dataset: {len(tokenized_pref)} samples")

In [ ]:
# DPO Configuration
dpo_config = DPOConfig(
    output_dir="./dpo-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-7,
    beta=0.1,  # KL regularization
    max_length=256,
    max_steps=50,
    logging_steps=10,
    report_to="none",
    fp16=False,
    warmup_ratio=0.1,
)

# DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=tokenized_pref,
    tokenizer=tokenizer,
)

print("Starting DPO training...")
dpo_trainer.train()

# Save model
dpo_trainer.save_model("./dpo-final")
print("DPO training complete!")

## 3. ORPO Training (No Reference Model)

In [ ]:
from trl import ORPOConfig, ORPOTrainer

# Load fresh model for ORPO
orpo_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

print("Model loaded for ORPO (no reference model needed!)")

In [ ]:
# ORPO Configuration
orpo_config = ORPOConfig(
    output_dir="./orpo-output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-6,
    beta=0.1,
    max_length=256,
    max_steps=50,
    logging_steps=10,
    report_to="none",
    fp16=False,
)

# ORPO Trainer (no ref_model!)
orpo_trainer = ORPOTrainer(
    model=orpo_model,
    args=orpo_config,
    train_dataset=tokenized_pref,
    tokenizer=tokenizer,
)

print("Starting ORPO training...")
orpo_trainer.train()

# Save model
orpo_trainer.save_model("./orpo-final")
print("ORPO training complete!")

## 4. Comparing SFT vs. DPO vs. ORPO Outputs

In [ ]:
from transformers import AutoModelForCausalLM

# Load all three models for comparison
print("Loading models for comparison...")

# Base SFT model
sft_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# DPO model
dpo_model = AutoModelForCausalLM.from_pretrained(
    "./dpo-final",
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

# ORPO model
orpo_model_final = AutoModelForCausalLM.from_pretrained(
    "./orpo-final",
    torch_dtype=torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

print("All models loaded!")

In [ ]:
def generate_comparison(prompt, models, tokenizer, max_length=100):
    """Generate responses from multiple models for comparison."""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    results = {}
    
    for name, model in models.items():
        model.eval()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_length,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        results[name] = generated
    
    return results

# Test prompts
test_prompts = [
    "How do I stay motivated?",
    "Explain quantum computing simply",
    "What's the best way to learn programming?",
]

models = {
    "SFT (Base)": sft_model,
    "DPO": dpo_model,
    "ORPO": orpo_model_final,
}

# Generate and compare
print("=" * 70)
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    print("-" * 70)
    
    results = generate_comparison(prompt, models, tokenizer)
    
    for name, output in results.items():
        # Extract just the response (after the prompt)
        if prompt in output:
            response = output.split(prompt)[-1].strip()
        else:
            response = output
        
        print(f"\n{name}:")
        print(f"  {response[:150]}..." if len(response) > 150 else f"  {response}")
    
    print("\n" + "=" * 70)

## 5. Preference Evaluation

In [ ]:
def pairwise_preference(model_a, model_b, prompt, tokenizer):
    """Generate A/B comparison for human evaluation."""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    
    outputs_a = model_a.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    outputs_b = model_b.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    
    response_a = tokenizer.decode(outputs_a[0], skip_special_tokens=True)
    response_b = tokenizer.decode(outputs_b[0], skip_special_tokens=True)
    
    print(f"Prompt: {prompt}")
    print("\n" + "-" * 50)
    print(f"Response A:\n{response_a}")
    print("\n" + "-" * 50)
    print(f"Response B:\n{response_b}")
    print("\n" + "-" * 50)
    print("Which response is better? (A/B)")
    
    return response_a, response_b

# Example A/B test
print("A/B Comparison Test")
print("=" * 50)
pairwise_preference(sft_model, dpo_model, "I'm feeling overwhelmed with work", tokenizer)

In [ ]:
# Automated preference evaluation using heuristics
def score_response_h heuristic(response):
    """Score response quality using simple heuristics."""
    score = 0
    
    # Length score (prefer medium-length responses)
    length = len(response.split())
    if 20 <= length <= 100:
        score += 1
    
    # Helpfulness indicators
    helpful_phrases = ["here are", "you can", "consider", "might help", "suggest"]
    if any(phrase in response.lower() for phrase in helpful_phrases):
        score += 1
    
    # Safety indicators (for sensitive topics)
    if any(word in response.lower() for word in ["professional", "support", "help"]):
        score += 1
    
    # Penalize overly short responses
    if length < 10:
        score -= 1
    
    return score

# Evaluate all models
eval_prompts = [
    "How do I deal with stress?",
    "Give me advice on learning a new skill",
    "What should I do if I'm unhappy at work?",
]

model_scores = {"SFT": [], "DPO": [], "ORPO": []}

for prompt in eval_prompts:
    results = generate_comparison(prompt, models, tokenizer)
    
    for name, output in results.items():
        response = output.split(prompt)[-1].strip() if prompt in output else output
        score = score_response_heuristic(response)
        model_scores[name.split()[0]].append(score)

# Plot scores
fig, ax = plt.subplots(figsize=(10, 5))
model_names = list(model_scores.keys())
avg_scores = [sum(scores)/len(scores) for scores in model_scores.values()]

bars = ax.bar(model_names, avg_scores, color=['#2d333b', '#238636', '#347d39'])
ax.set_ylabel('Average Heuristic Score')
ax.set_title('Model Comparison on Preference Heuristics')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, score in zip(bars, avg_scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
            f'{score:.2f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Summary

This notebook covered:

1. **Preference Dataset Creation**: Structure and analysis
2. **DPO Training**: Using TRL with reference model
3. **ORPO Training**: No reference model needed
4. **Model Comparison**: A/B testing SFT vs. DPO vs. ORPO
5. **Preference Evaluation**: Heuristic scoring

### Key Takeaways:

- DPO requires a reference model (usually SFT checkpoint)
- ORPO is faster (single model) with comparable results
- Use lower learning rates for alignment (5e-7 to 1e-6)
- A/B testing is essential for evaluating alignment
- Preference quality directly impacts alignment success